# Scorer weights from ratings (PLAN.md M7)

The ratings from `/admin/rate`, as `pnpm ratings:export` writes them (`data/ml/ratings.csv`: one row per rating, both rooms' features by today's scorer), walked through: what the ratings say before any fitting, then the Bradley–Terry fit that `fit_weights.py` runs, and how well each set of weights predicts homes it wasn't fitted on.

To change `packages/geometry/weights.json`, run the script, not this notebook: `uv run python fit_weights.py`.

In [ ]:
import json
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, '..')
import fit_weights as fw

weights = json.loads(fw.WEIGHTS.read_text())
feats = fw.room_features(weights)
w0 = np.array([weights['room'][f] for f in feats])
df = fw.load()
print(f"{len(df)} ratings, {df['home'].nunique()} homes, {df['room_id'].nunique()} room ids")

## What the ratings say, before any fitting

In [ ]:
summary = pd.DataFrame({
    'ratings': df.groupby('room_type').size(),
    'about the same': df[df.winner == 'same'].groupby('room_type').size(),
    'livable': df.groupby('room_type')['livable'].mean().round(2),
}).fillna(0)
summary

Which features the layouts picked have more of than those not picked, on average: a feature that winners clearly have more of deserves weight; one where they don't, less.

In [ ]:
decided = df[df.winner != 'same']
rows = []
for f in feats:
    a, b = decided[f'a_{f}'], decided[f'b_{f}']
    won = np.where(decided.winner == 'a', a, b)
    lost = np.where(decided.winner == 'a', b, a)
    both = ~(np.isnan(won) | np.isnan(lost))
    rows.append({'feature': f, 'pairs': int(both.sum()), 'winner minus loser': np.mean(won[both] - lost[both]) if both.any() else np.nan, 'hand-set weight': weights['room'][f]})
pd.DataFrame(rows).set_index('feature').round(3)

In [ ]:
fa, fb, y = fw.matrices(df, feats)
print('the hand-set scorer agrees with', f"{fw.accuracy(w0, fa, fb, y):.0%}", 'of the ratings with a winner')

## The fit

The chance A is preferred is `sigmoid(beta * (s_A - s_B))`, with `s` the room's score as the scorer works it out. The weights stay at or above zero and are pulled toward the hand-set ones.

In [ ]:
w, beta = fw.fit(fa, fb, y, w0)
w = fw.scaled(w, w0)
pd.DataFrame({'hand-set': w0, 'fitted': w.round(2)}, index=feats)

In [ ]:
cv = fw.cross_validate(df, feats, w0)
pd.DataFrame(cv).T.rename(columns={'loss': 'held-out log loss', 'acc': 'held-out accuracy'}).round(3)

## What the notes say

How often the notes, as read, blame each feature; and the problems they name that no feature measures, which need a feature before any weight can help.

In [ ]:
pd.Series(fw.blamed(df, feats), name='labels')

In [ ]:
for rule in fw.unmeasured():
    print('-', rule)